# Fink/LSST — Stacked histogram of alert counts vs Right Ascension

This notebook queries the **Fink broker REST API** (LSST endpoint) live,  
retrieves the latest alerts for several classification tags, and builds a  
**stacked histogram** of the number of *diaSources* as a function of RA.

The sky distribution directly reflects the **Rubin LSST pointing strategy**  
in effect on the query date.

---

**Column naming reminder (LSST DPDD schema)**

| Prefix | Meaning |
|--------|---------|
| `r:`   | diaSource table field (NOT the spectral band `r`) |
| `f:`   | Fink-computed field (classifiers, cross-matches) |

The spectral band is the *value* of column `r:band` ∈ `{u, g, r, i, z, y}`.

---

**Author:** dagoret  
**Date:** 2026-03-07  
**API:** https://api.lsst.fink-portal.org/api/v1

## 1 — Imports

In [ ]:
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timezone

print('Python imports OK')

## 2 — Dark-portal theme constants

In [ ]:
DARK_BG   = "#0d1117"
PANEL_BG  = "#161b22"
TEXT_COL  = "#e6edf3"
MUTED_COL = "#8b949e"
ACCENT    = "#58a6ff"
BORDER    = "#30363d"

plt.rcParams.update({
    'figure.facecolor'  : DARK_BG,
    'axes.facecolor'    : PANEL_BG,
    'axes.edgecolor'    : BORDER,
    'axes.labelcolor'   : TEXT_COL,
    'xtick.color'       : TEXT_COL,
    'ytick.color'       : TEXT_COL,
    'text.color'        : TEXT_COL,
    'grid.color'        : BORDER,
    'legend.facecolor'  : PANEL_BG,
    'legend.edgecolor'  : BORDER,
    'figure.dpi'        : 130,
})

## 3 — Configuration: tags and colours

Each entry is `(tag_name, hex_color, n_alerts_to_fetch)`.  
Increase `N_MAX` to probe a larger sky area.

In [ ]:
FINK_API = "https://api.lsst.fink-portal.org/api/v1"

# Tags to query  →  (display_label, bar_colour, n_to_fetch)
TAGS_CONFIG = [
    ("extragalactic_new_candidate",  "#58a6ff", 500),
    ("extragalactic_lt20mag_candidate","#3fb950",500),
    ("sn_near_galaxy_candidate",     "#ff7b72", 500),
    ("in_tns",                       "#ffa657", 300),
    ("hostless_candidate",           "#a371f7", 300),
]

# Columns to retrieve (keep minimal for speed)
COLUMNS = "r:diaObjectId,r:ra,r:band"

BIN_WIDTH_DEG = 2   # histogram bin width in degrees

## 4 — Fetch alerts from the Fink API

In [ ]:
def fetch_tag(tag: str, n: int, columns: str = COLUMNS) -> pd.DataFrame:
    """Query /api/v1/tags and return a DataFrame."""
    url    = f"{FINK_API}/tags"
    params = {"tag": tag, "n": n, "columns": columns, "output-format": "json"}
    resp   = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data   = resp.json()
    if not data:
        print(f"  [WARNING] tag '{tag}' returned 0 rows.")
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["tag"] = tag
    return df


frames = {}
query_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

for tag, colour, n_max in TAGS_CONFIG:
    print(f"Fetching  {n_max:4d}  diaSources  →  tag: {tag} ...", end=" ")
    try:
        df = fetch_tag(tag, n_max)
        frames[tag] = df
        print(f"{len(df):4d} rows")
    except Exception as exc:
        print(f"ERROR — {exc}")
        frames[tag] = pd.DataFrame()

print(f"\nQuery timestamp: {query_time}")

## 5 — Quick data summary

In [ ]:
for tag, colour, _ in TAGS_CONFIG:
    df = frames.get(tag, pd.DataFrame())
    if df.empty:
        print(f"{tag:<40s}  —  no data")
        continue
    ra = df["r:ra"]
    print(f"{tag:<40s}  N={len(df):4d}  "
          f"RA=[{ra.min():7.2f}° .. {ra.max():7.2f}°]  "
          f"median={ra.median():.2f}°")

## 6 — Build the stacked histogram

In [ ]:
# ── common bin edges across all tags ──────────────────────────────────────────
all_ra = np.concatenate([
    frames[tag]["r:ra"].values
    for tag, *_ in TAGS_CONFIG
    if not frames[tag].empty
])

ra_min = np.floor(all_ra.min() / BIN_WIDTH_DEG) * BIN_WIDTH_DEG
ra_max = np.ceil( all_ra.max() / BIN_WIDTH_DEG) * BIN_WIDTH_DEG
n_bins = int((ra_max - ra_min) / BIN_WIDTH_DEG)
bins   = np.linspace(ra_min, ra_max, n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
width       = bins[1] - bins[0]

print(f"Bins: {n_bins} bins  |  RA range: [{ra_min:.1f}°, {ra_max:.1f}°]  "
      f"|  bin width: {BIN_WIDTH_DEG}°")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

bottoms = np.zeros(n_bins)

for tag, colour, _ in TAGS_CONFIG:
    df = frames.get(tag, pd.DataFrame())
    if df.empty:
        continue
    counts, _ = np.histogram(df["r:ra"].values, bins=bins)
    ax.bar(
        bin_centers,
        counts,
        width   = width * 0.92,
        bottom  = bottoms,
        color   = colour,
        alpha   = 0.85,
        label   = f"{tag}  (N={len(df)})",
        edgecolor = DARK_BG,
        linewidth = 0.4,
    )
    bottoms += counts

# ── labels & cosmetics ────────────────────────────────────────────────────────
ax.set_xlabel("Right Ascension  [deg]", fontsize=13)
ax.set_ylabel("Number of diaSources",   fontsize=13)
ax.set_title(
    f"Fink/LSST alert distribution vs RA\n"
    f"(live data — {query_time}, {BIN_WIDTH_DEG}° bins)",
    fontsize=14, pad=12,
)

ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
ax.xaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(axis="y", linewidth=0.5, linestyle="--", alpha=0.6)
ax.grid(axis="x", linewidth=0.3, linestyle=":" ,  alpha=0.4)

# ── annotate known pointing fields ────────────────────────────────────────────
field_labels = {
    62.0:  "EDFS\nfield",
    53.0:  "WFD\nsouth",
    150.0: "COSMOS\n/ WFD",
    186.5: "Virgo\nstrip",
}
ymax_plot = bottoms.max()
for ra_lbl, txt in field_labels.items():
    if ra_min <= ra_lbl <= ra_max:
        ax.annotate(
            txt,
            xy=(ra_lbl, ymax_plot * 0.78),
            ha="center", va="bottom",
            color=MUTED_COL, fontsize=8.5,
        )

ax.legend(fontsize=10, loc="upper left")
fig.tight_layout(pad=1.5)
plt.show()

## 7 — Save the figure

In [ ]:
outpath = "fink_ra_stacked_histogram.png"
fig.savefig(outpath, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
print(f"Figure saved → {outpath}")

## 8 — (Optional) Per-tag RA distributions — side-by-side panels

In [ ]:
active_tags = [(tag, colour) for tag, colour, _ in TAGS_CONFIG
               if not frames.get(tag, pd.DataFrame()).empty]

n_cols = 2
n_rows = int(np.ceil(len(active_tags) / n_cols))

fig2, axes = plt.subplots(n_rows, n_cols,
                           figsize=(14, 4 * n_rows),
                           sharex=True, sharey=False)
axes = np.array(axes).ravel()

for i, (tag, colour) in enumerate(active_tags):
    df     = frames[tag]
    counts, _ = np.histogram(df["r:ra"].values, bins=bins)
    ax_i   = axes[i]
    ax_i.bar(bin_centers, counts, width=width * 0.92,
             color=colour, alpha=0.85,
             edgecolor=DARK_BG, linewidth=0.4)
    ax_i.set_title(f"{tag}  (N={len(df)})", fontsize=10)
    ax_i.set_ylabel("diaSources")
    ax_i.xaxis.set_major_locator(mticker.MultipleLocator(10))
    ax_i.grid(axis="y", linewidth=0.4, linestyle="--", alpha=0.5)

# hide unused panels
for j in range(len(active_tags), len(axes)):
    axes[j].set_visible(False)

for ax_i in axes[-n_cols:]:
    ax_i.set_xlabel("RA  [deg]")

fig2.suptitle("Per-tag RA distributions — Fink/LSST",
              fontsize=13, y=1.01)
fig2.tight_layout()
plt.show()

## 9 — (Optional) Stacked histogram coloured by spectral band

Here we stack on the *band* (`r:band`) dimension instead of tag,  
to see which Rubin filters contribute most at each RA.

In [ ]:
BAND_COLORS = {
    "u": "#2c7bb6",   # blue
    "g": "#1a9641",   # green
    "r": "#d7191c",   # red
    "i": "#fdae61",   # orange-yellow
    "z": "#7b3294",   # purple
    "y": "#c51b7d",   # magenta
}

MARKERS = {
    "u": "o",
    "g": "s",
    "r": "D",
    "i": "^",
    "z": "v",
    "y": "P",
}

# merge all tags into one DataFrame
df_all = pd.concat(
    [frames[tag] for tag, *_ in TAGS_CONFIG if not frames[tag].empty],
    ignore_index=True,
).drop_duplicates(subset="r:diaObjectId")

print(f"Total unique diaObjects across all tags: {len(df_all)}")

fig3, ax3 = plt.subplots(figsize=(14, 6))

bottoms3 = np.zeros(n_bins)
for band, bcolour in BAND_COLORS.items():
    sub = df_all[df_all["r:band"] == band]
    if sub.empty:
        continue
    counts_b, _ = np.histogram(sub["r:ra"].values, bins=bins)
    ax3.bar(bin_centers, counts_b,
            width=width * 0.92,
            bottom=bottoms3,
            color=bcolour, alpha=0.85,
            label=f"band {band}  (N={len(sub)})",
            edgecolor=DARK_BG, linewidth=0.4)
    bottoms3 += counts_b

ax3.set_xlabel("Right Ascension  [deg]", fontsize=13)
ax3.set_ylabel("Unique diaObjects",       fontsize=13)
ax3.set_title(
    f"Fink/LSST — alert count vs RA, stacked by Rubin band\n"
    f"({query_time}, all tags merged, {BIN_WIDTH_DEG}° bins)",
    fontsize=13, pad=10,
)
ax3.xaxis.set_major_locator(mticker.MultipleLocator(10))
ax3.grid(axis="y", linewidth=0.4, linestyle="--", alpha=0.5)
ax3.legend(fontsize=10, loc="upper left", ncol=2)
fig3.tight_layout()
plt.show()